# 58 — JSON Structured Output
**Goal:** Get structured JSON from LLMs using response_format and Pydantic parsing.

## 1. Why Structured Output?

In [ ]:
print('''Problems with freeform LLM output:
- Inconsistent formats between calls
- Parsing errors from markdown/incomplete JSON
- Hallucinated field names

Solutions:
1. OpenAI response_format = json_object / json_schema
2. instructor library (Pydantic-based)
3. Outlines (constrained generation)
4. Prompt-only: "Respond in JSON: {skill: ..., years: ...}"''')

## 2. Using response_format

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import json

class SkillExtract(BaseModel):
    skill_name: str
    years_experience: int
    proficiency: str

# With a working API key, this would be:
# client = OpenAI(api_key="...", base_url="https://openrouter.ai/api/v1")
# response = client.chat.completions.create(
#     model="openai/gpt-4o-mini",
#     messages=[{"role": "user", "content": "Extract skill from: 5 years Python"}],
#     response_format={"type": "json_object"},
# )
# result = json.loads(response.choices[0].message.content)
# print(result)

print("Structured output schema defined:")
print(SkillExtract.model_json_schema(indent=2))
print("\nWith API key: send request with response_format=json_object")

## 3. Pydantic Parsing

In [ ]:
from pydantic import BaseModel, ValidationError
from typing import List, Optional

class ResumeSkill(BaseModel):
    name: str
    category: str
    years: Optional[int] = None

class ResumeExtraction(BaseModel):
    skills: List[ResumeSkill]
    total_years: int

# Test parsing
valid_json = '{"skills": [{"name": "Python", "category": "technical"}], "total_years": 5}'
parsed = ResumeExtraction.model_validate_json(valid_json)
print(f"Parsed: {parsed.skills[0].name} -> {parsed.total_years} years")

# Error handling
try:
    bad_json = '{"skills": "Python"}'
    ResumeExtraction.model_validate_json(bad_json)
except ValidationError as e:
    print(f"\nValidation error handled gracefully:")
    print(f"  {e.errors()[0]['msg']}")

## Summary: Structured output is essential for production LLM use. Pydantic parsing catches errors at the boundary.